# 🏅 Olympics Data Analysis Dashboard
### SQL-Powered Cleaning & Analysis → Interactive Plotly Dash Dashboard

**Datasets (Kaggle):**
- 🥇 `piterfm/paris-2024-olympic-summer-games` — Paris 2024 medals, athletes, events
- 📜 `piterfm/olympic-games-medals-19862018` — Historical Olympics 1896–2022

---
## ⚙️ Step 1: Install Dependencies

In [1]:
!pip install kaggle dash plotly pandas --quiet
print('✅ All packages installed!')

✅ All packages installed!



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## 🔑 Step 2: Configure Kaggle API

1. Go to [kaggle.com](https://www.kaggle.com) → Profile → **Settings** → **API** → **Create New Token**
2. Upload the downloaded `kaggle.json` file below

In [2]:
import os
import shutil

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

source_path = os.path.join(os.getcwd(), 'kaggle.json')
target_path = os.path.join(kaggle_dir, 'kaggle.json')

if os.path.exists(source_path):
    shutil.copy2(source_path, target_path)
    try:
        os.chmod(target_path, 0o600)
    except Exception:
        pass
    print('✅ kaggle.json copied to ~/.kaggle/kaggle.json')
else:
    print('❌ kaggle.json not found in the workspace root')

✅ kaggle.json copied to ~/.kaggle/kaggle.json


---
## 📥 Step 3: Download Datasets from Kaggle

In [3]:
import kaggle

# authenticate programmatically
kaggle.api.authenticate()

# download Paris 2024 dataset
try:
    kaggle.api.dataset_download_files(
        'piterfm/paris-2024-olympic-summer-games',
        path='./paris2024', unzip=True, quiet=True
    )
    print('✅ Paris 2024 downloaded')
    print(os.listdir('./paris2024'))
except Exception as e:
    print(f'❌ Paris 2024 download failed: {e}')

# download historical dataset
try:
    kaggle.api.dataset_download_files(
        'piterfm/olympic-games-medals-19862018',
        path='./historical', unzip=True, quiet=True
    )
    print('✅ Historical data downloaded')
    print(os.listdir('./historical'))
except Exception as e:
    print(f'❌ Historical download failed: {e}')

Dataset URL: https://www.kaggle.com/datasets/piterfm/paris-2024-olympic-summer-games
✅ Paris 2024 downloaded
['athletes.csv', 'coaches.csv', 'events.csv', 'medallists.csv', 'medals.csv', 'medals_total.csv', 'nocs.csv', 'results', 'schedules.csv', 'schedules_preliminary.csv', 'teams.csv', 'technical_officials.csv', 'torch_route.csv', 'venues.csv']
Dataset URL: https://www.kaggle.com/datasets/piterfm/olympic-games-medals-19862018
✅ Historical data downloaded
['olympic_athletes.csv', 'olympic_hosts.csv', 'olympic_medals.csv', 'olympic_results.csv', 'olympic_results.pkl']


---
## 🗄️ Step 4: Load Raw Data into SQLite

In [4]:
import pandas as pd
import sqlite3
import os

conn = sqlite3.connect(':memory:', check_same_thread=False)
cur  = conn.cursor()

# load paris 2024 files
paris_dir = './paris2024'
for fname in os.listdir(paris_dir):
    if fname.endswith('.csv'):
        tbl = fname.replace('.csv','').replace('-','_').replace(' ','_').lower()
        df  = pd.read_csv(os.path.join(paris_dir, fname), encoding='latin1')
        df.to_sql(f'p24_{tbl}', conn, if_exists='replace', index=False)
        print(f'  ✅ Loaded: p24_{tbl}  ({len(df):,} rows)')

# load historical files
hist_dir = './historical'
for fname in os.listdir(hist_dir):
    if fname.endswith('.csv'):
        tbl = fname.replace('.csv','').replace('-','_').replace(' ','_').lower()
        df  = pd.read_csv(os.path.join(hist_dir, fname), encoding='latin1')
        df.to_sql(f'h_{tbl}', conn, if_exists='replace', index=False)
        print(f'  ✅ Loaded: h_{tbl}  ({len(df):,} rows)')

print('\n📋 All tables in DB:')
for row in cur.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall():
    print(' -', row[0])

# inspect columns so we know the exact names
print('=== Paris 2024 tables ===')
tables = [r[0] for r in cur.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name LIKE 'p24_%'"
).fetchall()]
for t in tables:
    cols = [c[1] for c in cur.execute(f'PRAGMA table_info({t})').fetchall()]
    print(f'\n  {t}: {cols}')

print('\n=== Historical tables ===')
tables_h = [r[0] for r in cur.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name LIKE 'h_%'"
).fetchall()]
for t in tables_h:
    cols = [c[1] for c in cur.execute(f'PRAGMA table_info({t})').fetchall()]
    print(f'\n  {t}: {cols}')

  ✅ Loaded: p24_athletes  (11,113 rows)
  ✅ Loaded: p24_coaches  (974 rows)
  ✅ Loaded: p24_events  (329 rows)
  ✅ Loaded: p24_medallists  (2,315 rows)
  ✅ Loaded: p24_medals  (1,044 rows)
  ✅ Loaded: p24_medals_total  (92 rows)
  ✅ Loaded: p24_nocs  (224 rows)
  ✅ Loaded: p24_schedules  (3,895 rows)
  ✅ Loaded: p24_schedules_preliminary  (2,298 rows)
  ✅ Loaded: p24_teams  (1,698 rows)
  ✅ Loaded: p24_technical_officials  (1,021 rows)
  ✅ Loaded: p24_torch_route  (73 rows)
  ✅ Loaded: p24_venues  (35 rows)
  ✅ Loaded: h_olympic_athletes  (75,904 rows)
  ✅ Loaded: h_olympic_hosts  (53 rows)
  ✅ Loaded: h_olympic_medals  (21,697 rows)
  ✅ Loaded: h_olympic_results  (162,804 rows)

📋 All tables in DB:
 - p24_athletes
 - p24_coaches
 - p24_events
 - p24_medallists
 - p24_medals
 - p24_medals_total
 - p24_nocs
 - p24_schedules
 - p24_schedules_preliminary
 - p24_teams
 - p24_technical_officials
 - p24_torch_route
 - p24_venues
 - h_olympic_athletes
 - h_olympic_hosts
 - h_olympic_medals
 -

---
## 🧹 Step 5: Data Cleaning — SQL Only

> **All cleaning done in SQL.** We build `CREATE VIEW` statements that:
> - Remove NULLs in key columns
> - Deduplicate rows
> - Standardise text (TRIM, UPPER, LOWER)
> - Filter invalid/unknown medal values
> - Unify column names for the historical table

In [5]:
# check table names before running views
print('Paris 2024 tables:')
for t in [r[0] for r in cur.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name LIKE 'p24_%'").fetchall()]:
    n = cur.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t}  ({n} rows)')

print('\nHistorical tables:')
for t in [r[0] for r in cur.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name LIKE 'h_%'").fetchall()]:
    n = cur.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t}  ({n} rows)')

# preview the historical medals table to identify column names
hist_tables = [r[0] for r in cur.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name LIKE 'h_%'").fetchall()]
if hist_tables:
    pd.read_sql_query(f'SELECT * FROM {hist_tables[0]} LIMIT 3', conn)

Paris 2024 tables:
  p24_athletes  (11113 rows)
  p24_coaches  (974 rows)
  p24_events  (329 rows)
  p24_medallists  (2315 rows)
  p24_medals  (1044 rows)
  p24_medals_total  (92 rows)
  p24_nocs  (224 rows)
  p24_schedules  (3895 rows)
  p24_schedules_preliminary  (2298 rows)
  p24_teams  (1698 rows)
  p24_technical_officials  (1021 rows)
  p24_torch_route  (73 rows)
  p24_venues  (35 rows)

Historical tables:
  h_olympic_athletes  (75904 rows)
  h_olympic_hosts  (53 rows)
  h_olympic_medals  (21697 rows)
  h_olympic_results  (162804 rows)


In [6]:
# clean historical view
# NOTE: The historical dataset (piterfm) uses columns:
#   discipline_title, event_title, slug_game, athlete_full_name,
#   country_name, country_code, medal_type, participant_type, gender

cur.execute('DROP VIEW IF EXISTS clean_historical')
cur.execute("""
CREATE VIEW clean_historical AS
WITH base AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY slug_game, athlete_full_name, event_title, medal_type
               ORDER BY rowid
           ) AS rn
    FROM h_olympic_medals
    WHERE athlete_full_name IS NOT NULL
      AND TRIM(athlete_full_name) != ''
      AND country_name       IS NOT NULL
      AND TRIM(country_name) != ''
      AND medal_type         IS NOT NULL
      AND UPPER(TRIM(medal_type)) IN ('GOLD', 'SILVER', 'BRONZE')
      AND discipline_title   IS NOT NULL
      AND slug_game          IS NOT NULL
)
SELECT
    TRIM(athlete_full_name)                     AS athlete_name,
    TRIM(country_name)                          AS country,
    UPPER(TRIM(country_code))                   AS country_code,
    UPPER(TRIM(medal_type))                     AS medal_type,
    TRIM(discipline_title)                      AS sport,
    TRIM(event_title)                           AS event,
    UPPER(TRIM(COALESCE(event_gender,'UNKNOWN')))     AS gender,
    UPPER(TRIM(participant_type))               AS participant_type,
    TRIM(slug_game)                             AS slug_game,
    -- Extract year and season from slug like 'paris-2024' or 'beijing-2022-winter'
    CAST(SUBSTR(slug_game, -4) AS INTEGER) AS game_year,
    CASE
        WHEN slug_game IN ('albertville-1992','calgary-1988','chamonix-1924',
                           'cortina-d-ampezzo-1956','garmisch-partenkirchen-1936',
                           'grenoble-1968','innsbruck-1964','innsbruck-1976',
                           'lake-placid-1932','lake-placid-1980','lillehammer-1994',
                           'nagano-1998','oslo-1952','salt-lake-city-2002',
                           'sarajevo-1984','sapporo-1972','squaw-valley-1960',
                           'st-moritz-1928','st-moritz-1948','turin-2006',
                           'vancouver-2010','beijing-2022','milan-cortina-2026')
        THEN 'Winter'
        ELSE 'Summer'
    END AS season
FROM base
WHERE rn = 1
""")
conn.commit()

n = pd.read_sql_query('SELECT COUNT(*) AS n FROM clean_historical', conn)['n'][0]
print(f'✅ clean_historical view created: {n:,} rows')
pd.read_sql_query('SELECT * FROM clean_historical LIMIT 5', conn)

✅ clean_historical view created: 18,072 rows


,athlete_name,country,country_code,medal_type,sport,event,gender,participant_type,slug_game,game_year,season
0,Adne SONDRAL,Norway,NO,SILVER,Speed skating,1500m men,MEN,ATHLETE,albertville-1992,1992,Winter
1,Alberto Tomba,Italy,IT,GOLD,Alpine Skiing,giant slalom men,MEN,ATHLETE,albertville-1992,1992,Winter
2,Alberto Tomba,Italy,IT,SILVER,Alpine Skiing,slalom men,MEN,ATHLETE,albertville-1992,1992,Winter
3,Anfisa REZTSOVA,Unified Team,None,GOLD,Biathlon,75km women,WOMEN,ATHLETE,albertville-1992,1992,Winter
4,Angelika NEUNER,Austria,AT,SILVER,Luge,Singles women,WOMEN,ATHLETE,albertville-1992,1992,Winter


In [7]:
# ── Clean VIEW: Paris 2024 Medals ────────────────────────────────────────
cur.execute('DROP VIEW IF EXISTS clean_paris2024')
cur.execute("""
CREATE VIEW clean_paris2024 AS
WITH base AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY name, discipline, event, medal_type
               ORDER BY rowid
           ) AS rn
    FROM p24_medallists
    WHERE name          IS NOT NULL AND TRIM(name) != ''
      AND country       IS NOT NULL AND TRIM(country) != ''
      AND medal_type    IS NOT NULL
      AND UPPER(TRIM(medal_type)) IN ('GOLD MEDAL','SILVER MEDAL','BRONZE MEDAL')
      AND discipline    IS NOT NULL
)
SELECT
    TRIM(name)                          AS athlete_name,
    TRIM(country)                       AS country,
    UPPER(TRIM(country_code))           AS country_code,
    CASE UPPER(TRIM(medal_type))
        WHEN 'GOLD MEDAL'   THEN 'GOLD'
        WHEN 'SILVER MEDAL' THEN 'SILVER'
        WHEN 'BRONZE MEDAL' THEN 'BRONZE'
        ELSE UPPER(TRIM(medal_type))
    END                                 AS medal_type,
    TRIM(discipline)                    AS sport,
    TRIM(event)                         AS event,
    UPPER(TRIM(COALESCE(gender,'UNKNOWN'))) AS gender,
    2024                                AS game_year,
    'Summer'                            AS season
FROM base
WHERE rn = 1
""")
conn.commit()

n = cur.execute('SELECT COUNT(*) FROM clean_paris2024').fetchone()[0]
print(f'clean_paris2024: {n} rows')

clean_paris2024: 2315 rows


In [8]:
# combine historical and Paris 2024 medal views
cur.execute('DROP VIEW IF EXISTS all_olympics')
cur.execute("""
CREATE VIEW all_olympics AS
    SELECT athlete_name, country, country_code, medal_type,
           sport, event, gender, game_year, season
    FROM clean_historical
    WHERE game_year < 2024

    UNION ALL

    SELECT athlete_name, country, country_code, medal_type,
           sport, event, gender, game_year, season
    FROM clean_paris2024
""")
conn.commit()

n = pd.read_sql_query('SELECT COUNT(*) AS n FROM all_olympics', conn)['n'][0]
print(f'✅ all_olympics union view: {n:,} total medal records')

✅ all_olympics union view: 20,387 total medal records


---
## 📊 Step 6: Data Analysis — SQL Only

All analysis is done purely with SQL.

In [9]:
# overall KPIs
kpi_df = pd.read_sql_query("""
SELECT
    COUNT(*)                                    AS total_medals,
    COUNT(DISTINCT athlete_name)                AS unique_athletes,
    COUNT(DISTINCT country)                     AS countries,
    COUNT(DISTINCT sport)                       AS sports,
    COUNT(DISTINCT game_year)                   AS olympic_editions,
    SUM(CASE WHEN medal_type='GOLD'   THEN 1 ELSE 0 END) AS gold_medals,
    SUM(CASE WHEN medal_type='SILVER' THEN 1 ELSE 0 END) AS silver_medals,
    SUM(CASE WHEN medal_type='BRONZE' THEN 1 ELSE 0 END) AS bronze_medals
FROM all_olympics
""", conn)
print('📌 Overall KPIs:')
kpi_df

📌 Overall KPIs:


,total_medals,unique_athletes,countries,sports,olympic_editions,gold_medals,silver_medals,bronze_medals
0,20387,14948,165,76,38,6639,6605,7143


In [10]:
# top 20 countries by medals
country_medals_df = pd.read_sql_query("""
SELECT
    country,
    SUM(CASE WHEN medal_type='GOLD'   THEN 1 ELSE 0 END) AS gold,
    SUM(CASE WHEN medal_type='SILVER' THEN 1 ELSE 0 END) AS silver,
    SUM(CASE WHEN medal_type='BRONZE' THEN 1 ELSE 0 END) AS bronze,
    COUNT(*) AS total
FROM all_olympics
GROUP BY country
ORDER BY gold DESC, silver DESC, bronze DESC
LIMIT 20
""", conn)
print('🏆 Top 20 Countries (All Time):')
country_medals_df

🏆 Top 20 Countries (All Time):


,country,gold,silver,bronze,total
0,United States of America,978,872,766,2616
1,Soviet Union,424,335,318,1077
2,Germany,317,374,345,1036
3,People's Republic of China,312,230,192,734
4,Great Britain,294,325,355,974
5,France,279,333,321,933
6,Italy,243,207,256,706
7,Japan,195,184,211,590
8,Netherlands,193,155,169,517
9,Norway,191,149,158,498


In [11]:
# medals over time
time_series_df = pd.read_sql_query("""
SELECT
    game_year,
    season,
    SUM(CASE WHEN medal_type='GOLD'   THEN 1 ELSE 0 END) AS gold,
    SUM(CASE WHEN medal_type='SILVER' THEN 1 ELSE 0 END) AS silver,
    SUM(CASE WHEN medal_type='BRONZE' THEN 1 ELSE 0 END) AS bronze,
    COUNT(*) AS total_medals,
    COUNT(DISTINCT country) AS countries_participating,
    COUNT(DISTINCT sport)   AS sports_count
FROM all_olympics
GROUP BY game_year, season
ORDER BY game_year
""", conn)
print('📅 Medals Over Time (sample):')
time_series_df.tail(10)

📅 Medals Over Time (sample):


,game_year,season,gold,silver,bronze,total_medals,countries_participating,sports_count
44,2006,Winter,76,76,76,228,26,13
45,2008,Summer,282,282,333,897,85,33
46,2010,Winter,78,79,77,234,26,13
47,2012,Summer,282,284,331,897,82,33
48,2014,Summer,88,85,88,261,26,13
49,2016,Summer,287,287,339,913,83,34
50,2018,Summer,93,89,92,274,29,14
51,2020,Summer,316,314,377,1007,90,37
52,2022,Winter,98,96,96,290,27,14
53,2024,Summer,752,756,807,2315,92,45


In [12]:
# top sports by medal count
sports_df = pd.read_sql_query("""
SELECT
    sport,
    COUNT(*) AS total_medals,
    COUNT(DISTINCT country) AS countries,
    COUNT(DISTINCT athlete_name) AS athletes,
    SUM(CASE WHEN medal_type='GOLD' THEN 1 ELSE 0 END) AS gold
FROM all_olympics
GROUP BY sport
ORDER BY total_medals DESC
LIMIT 20
""", conn)
print('🏋️ Top 20 Sports by Medals:')
sports_df

🏋️ Top 20 Sports by Medals:


,sport,total_medals,countries,athletes,gold
0,Athletics,3026,107,2287,1010
1,Swimming,1664,59,1016,556
2,Wrestling,1428,74,1173,446
3,Boxing,1048,90,960,278
4,Shooting,833,72,619,277
5,Gymnastics Artistic,820,34,374,284
6,Rowing,794,41,664,264
7,Canoe Sprint,783,44,499,260
8,Weightlifting,709,74,564,238
9,Judo,708,62,569,179


In [13]:
# gender distribution over time
gender_df = pd.read_sql_query("""
SELECT
    game_year,
    gender,
    COUNT(*) AS medals
FROM all_olympics
WHERE gender IN ('MALE','FEMALE','MEN','WOMEN','M','F')
GROUP BY game_year, gender
ORDER BY game_year
""", conn)

# normalize gender labels with SQL
gender_norm_df = pd.read_sql_query("""
SELECT
    game_year,
    CASE
        WHEN UPPER(gender) IN ('MALE','MEN','M')    THEN 'Male'
        WHEN UPPER(gender) IN ('FEMALE','WOMEN','W','F') THEN 'Female'
        ELSE 'Mixed/Other'
    END AS gender_group,
    COUNT(*) AS medals
FROM all_olympics
GROUP BY game_year, gender_group
ORDER BY game_year
""", conn)
print('👫 Gender distribution sample:')
gender_norm_df.tail(10)

👫 Gender distribution sample:


,game_year,gender_group,medals
100,2018,Male,129
101,2018,Mixed/Other,24
102,2020,Female,459
103,2020,Male,491
104,2020,Mixed/Other,57
105,2022,Female,126
106,2022,Male,132
107,2022,Mixed/Other,32
108,2024,Female,1162
109,2024,Male,1153


In [14]:
# Paris 2024 country medal table
paris_country_df = pd.read_sql_query("""
SELECT
    country,
    SUM(CASE WHEN medal_type='GOLD'   THEN 1 ELSE 0 END) AS gold,
    SUM(CASE WHEN medal_type='SILVER' THEN 1 ELSE 0 END) AS silver,
    SUM(CASE WHEN medal_type='BRONZE' THEN 1 ELSE 0 END) AS bronze,
    COUNT(*) AS total
FROM clean_paris2024
GROUP BY country
ORDER BY gold DESC, silver DESC, bronze DESC
LIMIT 20
""", conn)
print('🥇 Paris 2024 — Top 20 Countries:')
paris_country_df

🥇 Paris 2024 — Top 20 Countries:


,country,gold,silver,bronze,total
0,United States,134,101,95,330
1,China,71,57,40,168
2,Netherlands,67,25,26,118
3,France,53,95,39,187
4,Great Britain,40,42,80,162
5,Spain,40,7,36,83
6,Australia,33,45,45,123
7,Italy,31,29,28,88
8,Japan,27,31,24,82
9,New Zealand,26,18,7,51


In [15]:
# Paris 2024 top sports
paris_sports_df = pd.read_sql_query("""
SELECT
    sport,
    COUNT(*) AS medals,
    COUNT(DISTINCT country) AS countries
FROM clean_paris2024
GROUP BY sport
ORDER BY medals DESC
LIMIT 15
""", conn)
print('🏊 Paris 2024 — Sports:')
paris_sports_df

🏊 Paris 2024 — Sports:


,sport,medals,countries
0,Athletics,231,43
1,Swimming,219,19
2,Rowing,144,16
3,Football,124,6
4,Judo,105,26
5,Hockey,102,5
6,Handball,94,5
7,Fencing,90,13
8,Cycling Track,87,12
9,Water Polo,78,6


In [16]:
# summer vs winter medals over time
season_df = pd.read_sql_query("""
SELECT
    game_year,
    season,
    COUNT(*) AS medals,
    COUNT(DISTINCT country) AS countries
FROM all_olympics
GROUP BY game_year, season
ORDER BY game_year
""", conn)
print('☀️❄️ Season comparison:')
season_df.tail(10)

☀️❄️ Season comparison:


,game_year,season,medals,countries
44,2006,Winter,228,26
45,2008,Summer,897,85
46,2010,Winter,234,26
47,2012,Summer,897,82
48,2014,Summer,261,26
49,2016,Summer,913,83
50,2018,Summer,274,29
51,2020,Summer,1007,90
52,2022,Winter,290,27
53,2024,Summer,2315,92


In [17]:
# pre-fetch distinct filter values for dropdowns
all_countries = pd.read_sql_query("""
    SELECT DISTINCT country FROM all_olympics
    WHERE country IS NOT NULL ORDER BY country
""", conn)['country'].tolist()

all_sports = pd.read_sql_query("""
    SELECT DISTINCT sport FROM all_olympics
    WHERE sport IS NOT NULL ORDER BY sport
""", conn)['sport'].tolist()

year_range = pd.read_sql_query("""
    SELECT MIN(game_year) AS min_year, MAX(game_year) AS max_year
    FROM all_olympics
""", conn).iloc[0]

print(f'Countries: {len(all_countries)}')
print(f'Sports: {len(all_sports)}')
print(f'Year range: {int(year_range["min_year"])} – {int(year_range["max_year"])}')

Countries: 165
Sports: 76
Year range: 1896 – 2024


---
## 📈 Step 7: Interactive Dashboard — Python (Display Only)

All SQL analysis is done. Python here is only used to **render** the Dash dashboard.

In [18]:
from dash import Dash, html, dcc, Input, Output, dash_table
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import threading

# design system
C = {
    'bg':     '#05070f',
    'card':   '#0d1117',
    'card2':  '#161b22',
    'border': '#21262d',
    'gold':   '#FFD700',
    'silver': '#C0C0C0',
    'bronze': '#CD7F32',
    'blue':   '#58a6ff',
    'green':  '#3fb950',
    'purple': '#bc8cff',
    'pink':   '#f778ba',
    'text':   '#e6edf3',
    'sub':    '#8b949e',
}
MEDAL_COLORS = {'GOLD': C['gold'], 'SILVER': C['silver'], 'BRONZE': C['bronze']}
PALETTE = [C['blue'], C['green'], C['purple'], C['pink'], C['gold'], '#ff7b72', '#79c0ff']

CARD = {'backgroundColor': C['card'], 'border': f'1px solid {C["border"]}',
        'borderRadius': '12px', 'padding': '20px', 'marginBottom': '16px'}

# helper functions
def theme(fig, title=''):
    fig.update_layout(
        title={'text': title, 'font': {'size': 13, 'color': C['text']}, 'x': 0.01},
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        font_color=C['text'], margin=dict(l=8, r=8, t=38, b=8),
        legend=dict(bgcolor='rgba(0,0,0,0)', font_size=11),
    )
    fig.update_xaxes(gridcolor=C['border'], showline=False, tickfont_size=10)
    fig.update_yaxes(gridcolor=C['border'], showline=False, tickfont_size=10)
    return fig

def kpi(label, value, color=C['blue'], sub=''):
    return html.Div([
        html.P(label, style={'color': C['sub'], 'fontSize': '11px', 'margin': '0 0 4px 0',
                             'textTransform': 'uppercase', 'letterSpacing': '1px'}),
        html.H2(value, style={'color': color, 'margin': '0', 'fontSize': '26px', 'fontWeight': '800'}),
        html.P(sub, style={'color': C['sub'], 'fontSize': '11px', 'margin': '4px 0 0 0'}),
    ], style={**CARD, 'flex': '1', 'minWidth': '130px', 'textAlign': 'center'})

In [19]:
# dropdown options
opt_country = [{'label': '🌍 All Countries', 'value': '__ALL__'}] + \
              [{'label': c, 'value': c} for c in all_countries]

opt_sport   = [{'label': '🏅 All Sports', 'value': '__ALL__'}] + \
              [{'label': s, 'value': s} for s in all_sports]

opt_medal   = [{'label': '🥇 All Medals', 'value': '__ALL__'},
               {'label': '🥇 Gold',         'value': 'GOLD'},
               {'label': '🥈 Silver',        'value': 'SILVER'},
               {'label': '🥉 Bronze',        'value': 'BRONZE'}]

opt_season  = [{'label': '☀️❄️ All Seasons', 'value': '__ALL__'},
               {'label': '☀️ Summer',          'value': 'Summer'},
               {'label': '❄️ Winter',           'value': 'Winter'}]

MIN_YEAR = int(year_range['min_year'])
MAX_YEAR = int(year_range['max_year'])

DD = {'backgroundColor': C['card2'], 'color': '#000',
      'borderRadius': '8px', 'fontSize': '13px'}

# app layout
app = Dash(__name__, suppress_callback_exceptions=True)
app.title = 'Olympics Dashboard'

kv = kpi_df.iloc[0]

app.index_string = app.index_string.replace(
    '</head>',
    '''<style>
        input[type=number] { 
            background-color: #161b22 !important;
            color: #e6edf3 !important;
            border: 1px solid #21262d !important;
            border-radius: 6px !important;
            text-align: center !important;
        }
        .dash-slider-container .rc-slider-mark-text {
            color: #8b949e !important;
            font-size: 10px !important;
        }
    </style></head>'''
)

app.layout = html.Div(style={'backgroundColor': C['bg'], 'minHeight': '100vh',
                              'fontFamily': '"Inter","Segoe UI",sans-serif',
                              'color': C['text'], 'padding': '20px 28px'}, children=[

    # header
    html.Div([
        html.Div([
            html.H1('🏅 Olympics Analytics Dashboard',
                    style={'margin': '0', 'fontSize': '22px', 'fontWeight': '800',
                           'background': 'linear-gradient(90deg,#FFD700,#58a6ff)',
                           '-webkit-background-clip': 'text',
                           '-webkit-text-fill-color': 'transparent'}),
            html.P('SQL-powered analysis · Historical 1896–2024 · Plotly Dash',
                   style={'color': C['sub'], 'margin': '3px 0 0 0', 'fontSize': '12px'})
        ])
    ], style={'marginBottom': '20px'}),

    # kpi row
    html.Div([
        kpi('Total Medals',    f"{kv['total_medals']:,}",    C['blue'],   'All-time'),
        kpi('Athletes',        f"{kv['unique_athletes']:,}", C['green'],  'Unique'),
        kpi('Countries',       f"{kv['countries']:,}",       C['purple'], 'Participated'),
        kpi('Sports',          f"{kv['sports']:,}",          C['pink'],   'Disciplines'),
        kpi('Editions',        f"{kv['olympic_editions']:,}",C['gold'],   'Games held'),
        kpi('Gold Medals',     f"{kv['gold_medals']:,}",     C['gold'],   '🥇'),
        kpi('Silver Medals',   f"{kv['silver_medals']:,}",   C['silver'], '🥈'),
        kpi('Bronze Medals',   f"{kv['bronze_medals']:,}",   C['bronze'], '🥉'),
    ], style={'display': 'flex', 'gap': '12px', 'flexWrap': 'wrap', 'marginBottom': '16px'}),

    # filter bar
    html.Div([
        html.Div([
            html.Label('📅 Year Range', style={'color': C['sub'], 'fontSize': '11px',
                                               'display': 'block', 'marginBottom': '6px'}),
            dcc.RangeSlider(
                id='year-slider', min=MIN_YEAR, max=MAX_YEAR, step=4,
                value=[1896, 2024],
                marks={y: {'label': str(y), 'style': {'color': C['sub'], 'fontSize': '10px', 'fontWeight': '500'}}
                       for y in range(1896, 2025, 20)},
                updatemode='mouseup',
                className='custom-slider',
            )
        ], style={'flex': '3', 'minWidth': '280px'}),

        html.Div([
            html.Label('🌍 Country', style={'color': C['sub'], 'fontSize': '11px',
                                            'display': 'block', 'marginBottom': '6px'}),
            dcc.Dropdown(id='country-dd', options=opt_country,
                         value='__ALL__', style=DD, clearable=False)
        ], style={'flex': '2', 'minWidth': '160px'}),

        html.Div([
            html.Label('🏋️ Sport', style={'color': C['sub'], 'fontSize': '11px',
                                          'display': 'block', 'marginBottom': '6px'}),
            dcc.Dropdown(id='sport-dd', options=opt_sport,
                         value='__ALL__', style=DD, clearable=False)
        ], style={'flex': '2', 'minWidth': '160px'}),

        html.Div([
            html.Label('🥇 Medal', style={'color': C['sub'], 'fontSize': '11px',
                                          'display': 'block', 'marginBottom': '6px'}),
            dcc.Dropdown(id='medal-dd', options=opt_medal,
                         value='__ALL__', style=DD, clearable=False)
        ], style={'flex': '1', 'minWidth': '130px'}),

        html.Div([
            html.Label('☀️❄️ Season', style={'color': C['sub'], 'fontSize': '11px',
                                             'display': 'block', 'marginBottom': '6px'}),
            dcc.Dropdown(id='season-dd', options=opt_season,
                         value='__ALL__', style=DD, clearable=False)
        ], style={'flex': '1', 'minWidth': '130px'}),
    ], style={**CARD, 'display': 'flex', 'gap': '20px', 'alignItems': 'flex-end',
              'flexWrap': 'wrap'}),

    # row 1
    html.Div([
        html.Div([dcc.Graph(id='timeline-chart')], style={**CARD, 'flex': '2'}),
        html.Div([dcc.Graph(id='country-pie')],    style={**CARD, 'flex': '1'}),
    ], style={'display': 'flex', 'gap': '16px'}),

    # row 2
    html.Div([
        html.Div([dcc.Graph(id='country-bar')], style={**CARD, 'flex': '1'}),
        html.Div([dcc.Graph(id='sport-bar')],   style={**CARD, 'flex': '1'}),
    ], style={'display': 'flex', 'gap': '16px'}),

    # row 3
    html.Div([
        html.Div([dcc.Graph(id='gender-chart')],  style={**CARD, 'flex': '1'}),
        html.Div([dcc.Graph(id='season-chart')],  style={**CARD, 'flex': '1'}),
        html.Div([dcc.Graph(id='medal-donut')],   style={**CARD, 'flex': '1'}),
    ], style={'display': 'flex', 'gap': '16px'}),

    # row 4
    html.Div([
        html.H3('🗼 Paris 2024 Spotlight',
                style={'color': C['gold'], 'margin': '8px 0 12px 0', 'fontSize': '15px'}),
        html.Div([
            html.Div([dcc.Graph(id='paris-country')], style={**CARD, 'flex': '1'}),
            html.Div([dcc.Graph(id='paris-sport')],   style={**CARD, 'flex': '1'}),
        ], style={'display': 'flex', 'gap': '16px'}),
    ]),

    # footer
    html.P('Data: Kaggle / piterfm | Cleaned & analysed with SQL | Dashboard: Plotly Dash',
           style={'textAlign': 'center', 'color': C['sub'], 'fontSize': '11px', 'marginTop': '8px'})
])


# callback
@app.callback(
    Output('timeline-chart', 'figure'),
    Output('country-pie',    'figure'),
    Output('country-bar',    'figure'),
    Output('sport-bar',      'figure'),
    Output('gender-chart',   'figure'),
    Output('season-chart',   'figure'),
    Output('medal-donut',    'figure'),
    Output('paris-country',  'figure'),
    Output('paris-sport',    'figure'),
    Input('year-slider',  'value'),
    Input('country-dd',   'value'),
    Input('sport-dd',     'value'),
    Input('medal-dd',     'value'),
    Input('season-dd',    'value'),
)
def update(year_range_val, country, sport, medal, season):
    # build parameterized query filters
    y0, y1 = year_range_val
    params = [y0, y1]
    clauses = ['game_year BETWEEN ? AND ?']

    if country != '__ALL__':
        clauses.append('country = ?')
        params.append(country)
    if sport != '__ALL__':
        clauses.append('sport = ?')
        params.append(sport)
    if medal != '__ALL__':
        clauses.append('medal_type = ?')
        params.append(medal)
    if season != '__ALL__':
        clauses.append('season = ?')
        params.append(season)

    where = 'WHERE ' + ' AND '.join(clauses)

    tl = pd.read_sql_query(f'''
        SELECT game_year, medal_type, COUNT(*) AS medals
        FROM all_olympics {where}
        GROUP BY game_year, medal_type ORDER BY game_year
    ''', conn, params=params)

    cbar = pd.read_sql_query(f'''
        SELECT country,
            SUM(CASE WHEN medal_type='GOLD'   THEN 1 ELSE 0 END) AS gold,
            SUM(CASE WHEN medal_type='SILVER' THEN 1 ELSE 0 END) AS silver,
            SUM(CASE WHEN medal_type='BRONZE' THEN 1 ELSE 0 END) AS bronze,
            COUNT(*) AS total
        FROM all_olympics {where}
        GROUP BY country ORDER BY gold DESC, silver DESC LIMIT 15
    ''', conn, params=params)

    sprt = pd.read_sql_query(f'''
        SELECT sport, COUNT(*) AS medals,
               COUNT(DISTINCT country) AS countries
        FROM all_olympics {where}
        GROUP BY sport ORDER BY medals DESC LIMIT 15
    ''', conn, params=params)

    gen = pd.read_sql_query(f'''
        SELECT game_year,
            CASE WHEN UPPER(gender) IN ('MALE','MEN','M')         THEN 'Male'
                 WHEN UPPER(gender) IN ('FEMALE','WOMEN','W','F')  THEN 'Female'
                 ELSE 'Mixed/Other'
            END AS g, COUNT(*) AS medals
        FROM all_olympics {where}
        GROUP BY game_year, g ORDER BY game_year
    ''', conn, params=params)

    seas = pd.read_sql_query(f'''
        SELECT game_year, season, COUNT(*) AS medals
        FROM all_olympics {where}
        GROUP BY game_year, season ORDER BY game_year
    ''', conn, params=params)

    mtype = pd.read_sql_query(f'''
        SELECT medal_type, COUNT(*) AS medals
        FROM all_olympics {where}
        GROUP BY medal_type
    ''', conn, params=params)

    # Paris 2024 (year filter applied but ignore other filters for spotlight)
    pc = pd.read_sql_query('''
        SELECT country,
            SUM(CASE WHEN medal_type='GOLD'   THEN 1 ELSE 0 END) AS gold,
            SUM(CASE WHEN medal_type='SILVER' THEN 1 ELSE 0 END) AS silver,
            SUM(CASE WHEN medal_type='BRONZE' THEN 1 ELSE 0 END) AS bronze
        FROM clean_paris2024
        GROUP BY country ORDER BY gold DESC, silver DESC LIMIT 15
    ''', conn)

    ps = pd.read_sql_query('''
        SELECT sport, COUNT(*) AS medals
        FROM clean_paris2024
        GROUP BY sport ORDER BY medals DESC LIMIT 15
    ''', conn)

    # build charts
    fig_tl = go.Figure()
    for m, col in [('GOLD', C['gold']), ('SILVER', C['silver']), ('BRONZE', C['bronze'])]:
        d = tl[tl['medal_type'] == m]
        fig_tl.add_trace(go.Bar(x=d['game_year'], y=d['medals'], name=m,
                                marker_color=col, opacity=0.9))
    fig_tl.update_layout(barmode='stack')
    theme(fig_tl, '📅 Medals Over Time')

    top_pie = cbar.head(8)
    fig_cpie = px.pie(top_pie, names='country', values='total', hole=0.45,
                      color_discrete_sequence=PALETTE)
    theme(fig_cpie, '🌍 Medal Share (Top 8)')

    cbar_s = cbar.sort_values('total')
    fig_cb = go.Figure()
    for m, col in [('gold', C['gold']), ('silver', C['silver']), ('bronze', C['bronze'])]:
        fig_cb.add_trace(go.Bar(y=cbar_s['country'], x=cbar_s[m], name=m.title(),
                                orientation='h', marker_color=col))
    fig_cb.update_layout(barmode='stack', height=420)
    theme(fig_cb, '🌍 Top 15 Countries — Medal Breakdown')

    sprt_s = sprt.sort_values('medals')
    fig_sp = px.bar(sprt_s, y='sport', x='medals', orientation='h',
                    color='countries', color_continuous_scale='Blues',
                    labels={'medals': 'Medals', 'countries': 'Countries'})
    fig_sp.update_layout(height=420)
    theme(fig_sp, '🏋️ Top 15 Sports by Medals')

    fig_gen = px.area(gen, x='game_year', y='medals', color='g',
                      color_discrete_map={'Male': C['blue'], 'Female': C['pink'],
                                         'Mixed/Other': C['purple']})
    theme(fig_gen, '👫 Gender Participation Over Time')

    fig_seas = px.line(seas, x='game_year', y='medals', color='season',
                       markers=True,
                       color_discrete_map={'Summer': C['gold'], 'Winter': C['blue']})
    theme(fig_seas, '☀️❄️ Summer vs Winter Medals')

    fig_md = px.pie(mtype, names='medal_type', values='medals', hole=0.6,
                    color='medal_type',
                    color_discrete_map={'GOLD': C['gold'], 'SILVER': C['silver'],
                                        'BRONZE': C['bronze']})
    theme(fig_md, '🥇 Medal Distribution')

    pc_s = pc.sort_values('gold', ascending=False)
    fig_pc = go.Figure()
    for m, col in [('gold', C['gold']), ('silver', C['silver']), ('bronze', C['bronze'])]:
        fig_pc.add_trace(go.Bar(x=pc_s['country'], y=pc_s[m], name=m.title(),
                                marker_color=col))
    fig_pc.update_layout(barmode='stack')
    theme(fig_pc, '🗼 Paris 2024 — Country Medal Table')

    ps_s = ps.sort_values('medals')
    fig_ps = px.bar(ps_s, y='sport', x='medals', orientation='h',
                    color='medals', color_continuous_scale='Oranges')
    theme(fig_ps, '🗼 Paris 2024 — Medals by Sport')

    return (fig_tl, fig_cpie, fig_cb, fig_sp,
            fig_gen, fig_seas, fig_md, fig_pc, fig_ps)


print('✅ Dashboard app defined!')

✅ Dashboard app defined!


In [20]:
app.run(debug=False, port=8050)

In [21]:
print(os.listdir('./paris2024/results'))

['3x3 Basketball.csv', 'Archery.csv', 'Artistic Gymnastics.csv', 'Artistic Swimming.csv', 'Athletics.csv', 'Badminton.csv', 'Basketball.csv', 'Beach Volleyball.csv', 'Boxing.csv', 'Breaking.csv', 'Canoe Slalom.csv', 'Canoe Sprint.csv', 'Cycling BMX Freestyle.csv', 'Cycling BMX Racing.csv', 'Cycling Mountain Bike.csv', 'Cycling Road.csv', 'Cycling Track.csv', 'Diving.csv', 'Equestrian.csv', 'Fencing.csv', 'Football.csv', 'Golf.csv', 'Handball.csv', 'Hockey.csv', 'Judo.csv', 'Marathon Swimming.csv', 'Modern Pentathlon.csv', 'Rhythmic Gymnastics.csv', 'Rowing.csv', 'Rugby Sevens.csv', 'Sailing.csv', 'Shooting.csv', 'Skateboarding.csv', 'Sport Climbing.csv', 'Surfing.csv', 'Swimming.csv', 'Table Tennis.csv', 'Taekwondo.csv', 'Tennis.csv', 'Trampoline Gymnastics.csv', 'Triathlon.csv', 'Volleyball.csv', 'Water Polo.csv', 'Weightlifting.csv', 'Wrestling.csv']


---
## 📌 Project Summary

| Layer | Tool | Purpose |
|---|---|---|
| **Data** | Kaggle API | Paris 2024 + Historical 1896–2022 |
| **Storage** | SQLite in-memory | Hosts all raw tables |
| **Cleaning** | SQL `CREATE VIEW` | Nulls, dedup, standardise, UNION |
| **Analysis** | SQL `SELECT` | KPIs, trends, rankings, breakdown |
| **Dashboard** | Python + Plotly Dash | 9 charts + 8 KPI cards |

### 🎛️ Interactive Filters:
- **📅 Year Range** — RangeSlider from 1896 to 2024
- **🌍 Country** — Any participating nation
- **🏋️ Sport** — Any discipline
- **🥇 Medal Type** — Gold / Silver / Bronze / All
- **☀️❄️ Season** — Summer / Winter / All

### 📊 Charts:
1. 📅 Medals Over Time (stacked bar)
2. 🌍 Medal Share by Country (donut)
3. 🌍 Top 15 Countries — stacked breakdown
4. 🏋️ Top 15 Sports by medals (color = country diversity)
5. 👫 Gender Participation Trend (area chart)
6. ☀️❄️ Summer vs Winter medals (line chart)
7. 🥇 Medal Type Distribution (donut)
8. 🗼 Paris 2024 Country Medal Table
9. 🗼 Paris 2024 Sports Breakdown